# Genome-wide Prediction → BigWig
Slide a window across BED regions or a full chromosome and write ES/MS/LS/G1/WRT BigWig tracks.

In [ ]:
import sys
import numpy as np
import torch
import pyBigWig
from pathlib import Path
from pyfaidx import Fasta

sys.path.insert(0, '/liaozizhuo/repli-ATAC-seq/')

from src.infer._utils import load_model, parse_bed, fetch_one_hot, rc_average
from src.data.data_utils import GenomeSequence

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECKPOINT  = '/liaozizhuo/repli-ATAC-seq/outputs/basenji2_wt/checkpoints/best_model.pt'
CONFIG      = '/liaozizhuo/repli-ATAC-seq/src/configs/transformer_wt.yaml'
FASTA       = '/liaozizhuo/repli-ATAC-seq/data/genomes/rice_all_genomes_v7.fasta'

# BED mode: predict for each row in BED file
# Chrom mode: sliding window over a chromosome
MODE        = 'chrom'          # 'bed' | 'chrom'
BED         = 'data/regions.bed'   # used when MODE='bed'
CHROM       = 'chr12'              # used when MODE='chrom'

BATCH_SIZE  = 32
RC_AVERAGE  = True           # average forward + reverse complement
OUTPUT_DIR  = 'output/predict'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model(CHECKPOINT, CONFIG, device)
window_size = cfg['data']['input_window_length']
genome = GenomeSequence(FASTA)
fa = Fasta(FASTA)
chrom_sizes = {c: len(fa[c]) for c in fa.keys()}
print(f'Model loaded on {device}, window={window_size}bp')

In [ ]:
TRACKS = ['ES', 'MS', 'LS', 'G1', 'WRT']
records = {t: [] for t in TRACKS}

RAW_BIN_SIZE  = window_size // 256          # 128 bp (before AvgPool1d(8))
BIN_SIZE      = RAW_BIN_SIZE * 8            # 1024 bp per output bin
CROP          = 16                          # bins cropped each side (at 128bp resolution)
CROP_BP       = CROP * RAW_BIN_SIZE         # 2048 bp genomic offset to first output bin
CENTER_START  = 14                          # center bin index in 28-bin output (28//2)
CENTER_BINS   = 1                           # bins the model predicts
OUT_SIZE      = CENTER_BINS * BIN_SIZE      # 1024 bp stride for gapless coverage

if MODE == 'bed':
    regions = list(parse_bed(BED))
else:
    clen = chrom_sizes[CHROM]
    regions = [
        (CHROM, s, s + window_size, f'{CHROM}:{s}-{s+window_size}', '+')
        for s in range(0, clen - window_size + 1, OUT_SIZE)
    ]

print(f'{len(regions):,} regions to predict  (stride={OUT_SIZE} bp, bin={BIN_SIZE} bp)')

In [ ]:
batch_oh, batch_meta = [], []

def flush(batch_oh, batch_meta):
    if not batch_oh:
        return
    x = torch.tensor(np.stack(batch_oh), dtype=torch.float32).to(device)
    if RC_AVERAGE:
        log1p_pred = rc_average(model, x).cpu().numpy()   # [B, 2, 4]
    else:
        with torch.no_grad():
            log1p_pred = model(x)['phase_pred'].cpu().numpy()

    B, T, C = log1p_pred.shape   # T = CENTER_BINS = 2
    pred = np.expm1(np.clip(log1p_pred, 0, None))         # [B, 2, 4]  TPM scale

    # Compute WRT directly from predicted TPM values — consistent with ground truth
    # which uses global-TPM ES/MS/LS. compute_wrt_from_phase_pred re-normalizes
    # within the batch, making values batch-dependent and incomparable across regions.
    eps = 1e-6
    es, ms, ls = pred[..., 0], pred[..., 1], pred[..., 2]
    wrt = (0.5 * ms + ls) / (es + ms + ls + eps)          # [B, 2]

    for i, (chrom, win_start, win_end) in enumerate(batch_meta):
        ws = (win_start + win_end) // 2 - window_size // 2
        for k in range(T):
            bs = ws + CROP_BP + (CENTER_START + k) * BIN_SIZE
            be = bs + BIN_SIZE
            for j, t in enumerate(['ES', 'MS', 'LS', 'G1']):
                records[t].append((chrom, bs, be, float(pred[i, k, j])))
            records['WRT'].append((chrom, bs, be, float(wrt[i, k])))

for chrom, start, end, name, strand in regions:
    oh = fetch_one_hot(genome, chrom, start, end, window_size, strand)
    batch_oh.append(oh)
    batch_meta.append((chrom, start, end))
    if len(batch_oh) == BATCH_SIZE:
        flush(batch_oh, batch_meta)
        batch_oh, batch_meta = [], []
flush(batch_oh, batch_meta)

print('Inference done')

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

for track, entries in records.items():
    entries.sort(key=lambda x: (x[0], x[1]))
    bw = pyBigWig.open(str(out_dir / f'{track}.bw'), 'w')
    bw.addHeader(list(chrom_sizes.items()))
    for chrom, start, end, val in entries:
        bw.addEntries([chrom], [start], ends=[end], values=[float(val)])
    bw.close()

print(f'Written {len(TRACKS)} BigWig tracks to {OUTPUT_DIR}')